# 01 — Class Distribution & Metadata EDA

Understand the class balance, metadata distributions, and galactic vs extragalactic split
in the PLAsTiCC training set.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG, VIS_CONFIG
from src.utils import get_class_name, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
print(f"Loaded {len(metadata)} objects")

## 1. Class Distribution

In [ ]:
# Class counts and percentages
class_counts = metadata['target'].value_counts().sort_index()
class_df = pd.DataFrame({
    'class_id': class_counts.index,
    'name': [get_class_name(c) for c in class_counts.index],
    'count': class_counts.values,
    'pct': 100 * class_counts.values / len(metadata),
    'type': ['Galactic' if c in DATA_CONFIG['galactic_classes'] else 'Extragalactic'
             for c in class_counts.index],
})
print(class_df.to_string(index=False))

In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2196F3' if c in DATA_CONFIG['galactic_classes'] else '#FF5722'
          for c in class_counts.index]
bars = ax.bar(range(len(class_counts)), class_counts.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels([f"{get_class_name(c)}\n({c})" for c in class_counts.index],
                    rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Count')
ax.set_yscale('log')
ax.set_title('PLAsTiCC Training Set Class Distribution')

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2196F3', label='Galactic'),
                    Patch(color='#FF5722', label='Extragalactic')])

# Count labels on bars
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            str(count), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_figure(fig, 'class_distribution')
plt.show()

## 2. Galactic vs Extragalactic Split

In [ ]:
metadata['source_type'] = metadata['target'].apply(
    lambda t: 'Galactic' if t in DATA_CONFIG['galactic_classes'] else 'Extragalactic'
)
print(metadata['source_type'].value_counts())

fig, ax = plt.subplots(figsize=(6, 6))
counts = metadata['source_type'].value_counts()
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=['#2196F3', '#FF5722'], startangle=90)
ax.set_title('Galactic vs Extragalactic')
plt.show()

## 3. Metadata Distributions by Class

In [ ]:
# Host galaxy photo-z distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in zip(axes.flat,
    ['hostgal_photoz', 'distmod', 'mwebv', 'hostgal_photoz_err'],
    ['Host Galaxy Photo-z', 'Distance Modulus', 'MW E(B-V)', 'Photo-z Error']):
    for target in sorted(DATA_CONFIG['class_names'].keys()):
        subset = metadata[metadata['target'] == target][col].dropna()
        if len(subset) > 0 and subset.std() > 0:
            ax.hist(subset, bins=30, alpha=0.4, label=get_class_name(target), density=True)
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.set_title(title)

# Put legend outside
axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
save_figure(fig, 'metadata_distributions')
plt.show()

## 4. Redshift Distribution

Key diagnostic: galactic classes should have `hostgal_specz = 0` (no host galaxy redshift).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for target in sorted(DATA_CONFIG['extragalactic_classes']):
    subset = metadata[metadata['target'] == target]['hostgal_specz']
    subset = subset[subset > 0]
    if len(subset) > 0:
        ax.hist(subset, bins=30, alpha=0.5, label=get_class_name(target), density=True)

ax.set_xlabel('Host Galaxy Spectroscopic Redshift')
ax.set_ylabel('Density')
ax.set_title('Redshift Distribution (Extragalactic Classes Only)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. DDF vs WFD

In [ ]:
# Deep Drilling Field flag
ddf_counts = metadata.groupby(['target', 'ddf']).size().unstack(fill_value=0)
ddf_counts.index = [f"{get_class_name(t)} ({t})" for t in ddf_counts.index]
ddf_counts.columns = ['WFD', 'DDF']
print(ddf_counts)
print(f"\nTotal DDF: {metadata['ddf'].sum()}, WFD: {(~metadata['ddf'].astype(bool)).sum()}")

In [ ]:
# Save class balance summary
class_df.to_csv(os.path.join(DATA_CONFIG['processed_dir'], 'class_balance.csv'), index=False)
print("Saved class balance summary")